# Jour 3 · Chapitre 2 — Scénario : le tableau de bord des flux pour la direction
## Formation Beobank · SAS → Python · Cahier participant

**L'histoire.** La direction des opérations veut suivre l'activité des comptes chaque mois.
Julien, responsable du reporting, vous confie le chantier :
*"Il me faut : les comptes les plus actifs, le poids de chaque mouvement dans son compte, les
mouvements anormaux par rapport au précédent, l'évolution mensuelle avec une moyenne glissante,
un graphique, et la répartition FR / NL. Tu fais le SQL, et tu vérifies avec pandas."*

**Comment travailler.** Remplacez chaque `______` — mots-clés SQL **et** méthodes pandas.

> ⚠️ `TXN_X_CTR` n'a pas de montant : comme dans le cours, `MNT_MVT` est **simulé** (graine 42)
> pour pouvoir calculer. Les montants sont fictifs, la méthode est réelle.

📎 **Matière à relire :** `Jour3_Formateur.ipynb` (même dossier) — Modules 1 (CTE), 2 (`OVER
PARTITION BY` / `.transform()`), 3 (`ROW_NUMBER`, `LAG` / `.shift()`), 4 (dates), 5 (fenêtre
glissante), 6 (Matplotlib), 7 (`resample`, `pct_change`, `rank`), 8 (écrire dans la base).

## Préparation (déjà écrite, exécutez-la)

In [ ]:
# ── Chargement + base SQLite en mémoire ───────────────────────────────────
import sqlite3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PARAMS = dict(sep=";", na_values=".", encoding="utf-8")
txn = pd.read_csv("../../data/TXN_X_CTR.csv", **PARAMS)

rng = np.random.default_rng(42)                                  # même graine que le cours
txn["DAT_MVT"] = pd.to_datetime(txn["DAT_CRE_MVT_CPB"])           # vraie date
txn["MNT_MVT"] = rng.normal(250, 400, size=len(txn)).round(2)    # montant SIMULÉ

conn = sqlite3.connect(":memory:")
txn.drop(columns="DAT_MVT").to_sql("TXN_X_CTR", conn, index=False)   # dates gardées en texte AAAA-MM-JJ

def sql(requete):
    """Exécute une requête sur la base et renvoie un DataFrame."""
    return pd.read_sql(requete, conn)

print(len(txn), "mouvements chargés")

---
## Mission 1 — Les comptes les plus actifs (CTE)

*"Top 5 des comptes par montant total des mouvements, avec leur nombre de mouvements."*

In [ ]:
# ── Common Table Expression ───────────────────────────────────────────────
print(sql("""
-- Quel mot-clé ouvre une CTE (une requête nommée réutilisable) ?
______ volumes AS (
    SELECT IDT_AC, COUNT(*) AS nb_mvt, ROUND(SUM(MNT_MVT), 2) AS total
    FROM TXN_X_CTR
    -- Quelle clause regroupe les lignes par compte ?
    ______ IDT_AC
)
SELECT * FROM volumes
-- Du plus grand total au plus petit : quel sens de tri ?
ORDER BY total ______
LIMIT 5
"""))

---
## Mission 2 — Le poids de chaque mouvement dans son compte

*"Pour chaque mouvement : le total de son compte, et la part qu'il représente."*

In [ ]:
# ── Fonction de fenêtre OVER (PARTITION BY) ───────────────────────────────
poids_sql = sql("""
SELECT IDT_AC, NUM_ORD_MVT_CPB, MNT_MVT,
       -- Quel mot-clé transforme SUM en fonction de fenêtre ? Quelle clause découpe par compte ?
       SUM(MNT_MVT) ______ (______ IDT_AC) AS total_compte
FROM TXN_X_CTR
""")
poids_sql["poids_pct"] = (100 * poids_sql["MNT_MVT"] / poids_sql["total_compte"]).round(1)
print(poids_sql.head())

# ── Vérification pandas ───────────────────────────────────────────────────
# Quelle méthode de groupby renvoie un résultat de MÊME taille que le tableau (comme OVER) ?
txn["total_compte"] = txn.groupby("IDT_AC")["MNT_MVT"].______("sum")
print("SQL et pandas identiques :",
      np.allclose(sorted(poids_sql["total_compte"]), sorted(txn["total_compte"])))   # → True

---
## Mission 3 — Les mouvements anormaux

*"Un mouvement qui dépasse de plus de 500 € le précédent du même compte, je veux le voir."*

In [ ]:
# ── ROW_NUMBER et LAG ─────────────────────────────────────────────────────
anomalies = sql("""
WITH ordonnes AS (
    SELECT IDT_AC, DAT_CRE_MVT_CPB, NUM_ORD_MVT_CPB, MNT_MVT,
           -- Quelle fonction numérote les lignes 1, 2, 3... dans chaque compte ?
           ______() OVER (PARTITION BY IDT_AC ORDER BY DAT_CRE_MVT_CPB, NUM_ORD_MVT_CPB) AS rang,
           -- Quelle fonction lit la valeur de la ligne PRÉCÉDENTE ?
           ______(MNT_MVT) OVER (PARTITION BY IDT_AC ORDER BY DAT_CRE_MVT_CPB, NUM_ORD_MVT_CPB) AS mnt_precedent
    FROM TXN_X_CTR
)
SELECT * FROM ordonnes
WHERE MNT_MVT - mnt_precedent > 500
""")
print(len(anomalies), "mouvements anormaux")

# ── Vérification pandas ───────────────────────────────────────────────────
t = txn.sort_values(["IDT_AC", "DAT_CRE_MVT_CPB", "NUM_ORD_MVT_CPB"])
# Quelle méthode décale une colonne d'une ligne vers le bas (équivalent de LAG) ?
t["mnt_precedent"] = t.groupby("IDT_AC")["MNT_MVT"].______(1)
# Quelle méthode numérote les lignes de chaque groupe à partir de 0 (ROW_NUMBER - 1) ?
t["rang"] = t.groupby("IDT_AC").______() + 1
print("pandas trouve :", ((t["MNT_MVT"] - t["mnt_precedent"]) > 500).sum())

---
## Mission 4 — Le volume par mois

*"Combien de mouvements et quel montant, mois par mois ?"*

In [ ]:
# ── Dates en SQL ──────────────────────────────────────────────────────────
print(sql("""
-- strftime extrait une partie de date : quel format donne "2026-03" (année-mois) ?
SELECT strftime('______', DAT_CRE_MVT_CPB) AS mois,
       COUNT(*) AS nb_mvt,
       ROUND(SUM(MNT_MVT), 2) AS total
FROM TXN_X_CTR
GROUP BY mois
ORDER BY mois
"""))

---
## Mission 5 — Tendance et moyenne glissante (pandas)

In [ ]:
# ── resample, pct_change, rolling, rank ───────────────────────────────────
# Quelle méthode regroupe une série datée par période (ici "ME" = fin de mois) ?
mensuel = txn.set_index("DAT_MVT")["MNT_MVT"].______("ME").sum()
synthese = mensuel.to_frame("total")
# Quelle méthode calcule la variation en % par rapport à la ligne précédente ?
synthese["variation_pct"] = (synthese["total"].______() * 100).round(1)
# Quelle méthode crée une fenêtre glissante de 3 mois ?
synthese["moyenne_3m"] = synthese["total"].______(3).mean().round(2)
# Quelle méthode classe les mois (1 = meilleur mois) ?
synthese["rang"] = synthese["total"].______(ascending=False).astype(int)
synthese

---
## Mission 6 — Le graphique de la direction

In [ ]:
# ── Matplotlib ────────────────────────────────────────────────────────────
# Quelle fonction crée une figure ET un système d'axes en une fois ?
fig, ax = plt.______(figsize=(9, 4))
ax.bar(synthese.index.strftime("%Y-%m"), synthese["total"], color="#9DB8D9", label="Total du mois")
# Quelle méthode de l'axe trace une COURBE ?
ax.______(synthese.index.strftime("%Y-%m"), synthese["moyenne_3m"], color="#C0392B",
          marker="o", label="Moyenne 3 mois")
ax.set_title("Flux mensuels (montants simulés)")
ax.legend()
plt.show()

---
## Mission 7 — Déposer la synthèse dans la base

In [ ]:
# ── Écrire dans la base ───────────────────────────────────────────────────
a_ecrire = synthese.reset_index().assign(DAT_MVT=lambda d: d["DAT_MVT"].dt.strftime("%Y-%m"))
# Si la table existe déjà (rapport relancé chaque mois), que faire ? (valeur de if_exists)
a_ecrire.to_sql("SYNTHESE_MENSUELLE", conn, if_exists="______", index=False)
print(sql("SELECT COUNT(*) AS nb_mois FROM SYNTHESE_MENSUELLE"))

---
## 🏁 Mission finale — Francophones vs néerlandophones

*"Par mois : combien de relevés en FR, combien en NL, et la part de FR."*

Format attendu : une ligne par mois, colonnes `mois | nb_fr | nb_nl | part_fr_pct`.

In [ ]:
# ── CTE + CASE WHEN ───────────────────────────────────────────────────────
print(sql("""
WITH par_mois AS (
    SELECT strftime('%Y-%m', DAT_CRE_MVT_CPB) AS mois,
           -- Quel mot-clé SQL ouvre une condition dans un SELECT (SI ... ALORS ... SINON) ?
           SUM(______ WHEN COD_LNG_RIU = 'FR' THEN 1 ELSE 0 END) AS nb_fr,
           SUM(CASE WHEN COD_LNG_RIU = '______' THEN 1 ELSE 0 END) AS nb_nl
    FROM TXN_X_CTR
    GROUP BY mois
)
SELECT mois, nb_fr, nb_nl,
       ROUND(100.0 * nb_fr / (nb_fr + nb_nl), 1) AS part_fr_pct
FROM par_mois
ORDER BY mois
"""))

✍️ **Commentaire pour Julien (3 phrases) :** tendance des flux, mois à surveiller, poids du
néerlandais. Rappelez que les montants sont simulés.